# DUMBO CityGML to GeoJSON walkthrough

This notebook demonstrates the current narrow `nyc-mesh` v0.1 happy path on a small, reproducible fixture that approximates a DUMBO-scale clip.

What it does:

1. creates a tiny local CityGML file with a few building footprints
2. uses the shipped SDK helper to extract height-aware buildings
3. clips them with a WGS84 bounding box near DUMBO
4. exports a GeoJSON file that can be used in downstream notebook or web workflows

Current v0.1 limitations remain explicit:

- local CityGML files only
- only buildings with `bldg:measuredHeight`
- source coordinates assumed to be `EPSG:2263`
- output reprojected to `EPSG:4326`
- bbox clipping uses WGS84 extents


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

from nyc_mesh import BoundingBox, export_citygml_geojson, extract_citygml_buildings

In [ ]:
workspace = Path.cwd()
notebook_dir = workspace if workspace.name == "notebooks" else workspace / "notebooks"
fixture_path = notebook_dir / "dumbo-sample.gml"
output_path = notebook_dir / "dumbo-buildings.geojson"

fixture_path.write_text(
    """<?xml version="1.0" encoding="UTF-8"?>
<core:CityModel
  xmlns:core="http://www.opengis.net/citygml/2.0"
  xmlns:gml="http://www.opengis.net/gml"
  xmlns:bldg="http://www.opengis.net/citygml/building/2.0">
  <core:cityObjectMember>
    <bldg:Building gml:id="washington-street-inside">
      <bldg:measuredHeight>28.4</bldg:measuredHeight>
      <bldg:lod1Solid>
        <gml:Solid>
          <gml:exterior>
            <gml:CompositeSurface>
              <gml:surfaceMember>
                <gml:Polygon gml:id="poly-1">
                  <gml:exterior>
                    <gml:LinearRing>
                      <gml:posList srsDimension="3">
                        986350 195550 0 986430 195550 0 986430 195620 0 986350 195620 0 986350 195550 0
                      </gml:posList>
                    </gml:LinearRing>
                  </gml:exterior>
                </gml:Polygon>
              </gml:surfaceMember>
            </gml:CompositeSurface>
          </gml:exterior>
        </gml:Solid>
      </bldg:lod1Solid>
    </bldg:Building>
  </core:cityObjectMember>
  <core:cityObjectMember>
    <bldg:Building gml:id="water-street-inside">
      <bldg:measuredHeight>17.0</bldg:measuredHeight>
      <bldg:lod1Solid>
        <gml:Solid>
          <gml:exterior>
            <gml:CompositeSurface>
              <gml:surfaceMember>
                <gml:Polygon gml:id="poly-2">
                  <gml:exterior>
                    <gml:LinearRing>
                      <gml:posList srsDimension="3">
                        986480 195480 0 986560 195480 0 986560 195560 0 986480 195560 0 986480 195480 0
                      </gml:posList>
                    </gml:LinearRing>
                  </gml:exterior>
                </gml:Polygon>
              </gml:surfaceMember>
            </gml:CompositeSurface>
          </gml:exterior>
        </gml:Solid>
      </bldg:lod1Solid>
    </bldg:Building>
  </core:cityObjectMember>
  <core:cityObjectMember>
    <bldg:Building gml:id="outside-clip">
      <bldg:measuredHeight>32.0</bldg:measuredHeight>
      <bldg:lod1Solid>
        <gml:Solid>
          <gml:exterior>
            <gml:CompositeSurface>
              <gml:surfaceMember>
                <gml:Polygon gml:id="poly-3">
                  <gml:exterior>
                    <gml:LinearRing>
                      <gml:posList srsDimension="3">
                        988200 197400 0 988300 197400 0 988300 197500 0 988200 197500 0 988200 197400 0
                      </gml:posList>
                    </gml:LinearRing>
                  </gml:exterior>
                </gml:Polygon>
              </gml:surfaceMember>
            </gml:CompositeSurface>
          </gml:exterior>
        </gml:Solid>
      </bldg:lod1Solid>
    </bldg:Building>
  </core:cityObjectMember>
  <core:cityObjectMember>
    <bldg:Building gml:id="no-height">
      <bldg:lod1Solid>
        <gml:Solid>
          <gml:exterior>
            <gml:CompositeSurface>
              <gml:surfaceMember>
                <gml:Polygon gml:id="poly-4">
                  <gml:exterior>
                    <gml:LinearRing>
                      <gml:posList srsDimension="3">
                        986390 195430 0 986450 195430 0 986450 195490 0 986390 195490 0 986390 195430 0
                      </gml:posList>
                    </gml:LinearRing>
                  </gml:exterior>
                </gml:Polygon>
              </gml:surfaceMember>
            </gml:CompositeSurface>
          </gml:exterior>
        </gml:Solid>
      </bldg:lod1Solid>
    </bldg:Building>
  </core:cityObjectMember>
</core:CityModel>
""",
    encoding="utf-8",
)

fixture_path

In [ ]:
bbox = BoundingBox(
    min_lat=40.7005,
    min_lon=-73.9925,
    max_lat=40.7048,
    max_lon=-73.9865,
)

features = extract_citygml_buildings(fixture_path, bbox=bbox)
[(feature.building_id, round(feature.height, 1)) for feature in features]

## Export the clipped buildings to GeoJSON

The exported GeoJSON is the same shape produced by the CLI, but here it stays convenient for notebook-driven inspection and downstream analysis.


In [ ]:
exported = export_citygml_geojson(fixture_path, output_path, bbox=bbox)
payload = json.loads(exported.read_text(encoding="utf-8"))

{
    "output_path": str(exported),
    "feature_ids": [feature["id"] for feature in payload["features"]],
    "feature_count": len(payload["features"]),
}

## Inspect one exported feature

This is useful for validating the exact schema that downstream notebook, web-map, or ETL code will receive.


In [ ]:
payload["features"][0]